# Import libraries

In [1]:
import bokeh
import csv
import logging
import math
import matplotlib as plt
import numba
import numpy as np
import pandas as pd
import pymysql
import re
import seaborn as sns
from sklearn.metrics import mean_squared_error
import scipy
import sqlalchemy as db
import statsmodels

# Setup

In [2]:
# Set log level to DEBUG while developing
log_level = logging.DEBUG

# Read CSV

In [3]:
class FileReader():
    @staticmethod
    def read_csv_to_df(file=None, logger=None):
        output = None
        try:
            output = pd.read_csv(filepath_or_buffer=file)
            output.set_index('x', inplace=True)
            message = "File loaded: {file}"
        except FileNotFoundError as e:
            message = "File not found: {file}"
        except:
            message = "An unknown error occured with file: {file}."
        finally:
            if logger:
                logger.info(message.format(file=file))
        return output
    
    @staticmethod
    def csv_line_gen(file=None):
        if file:
            handler = open(file, "r")
            reader = csv.reader(handler)
            data_list = []
            for i, line in enumerate(reader):
                yield line


# Regression

In [91]:
class RegressionException(Exception):
    pass

class Regression():
    def __init__(self, ideal=None):
        if ideal is None:
            raise RegressionException
        self._ideal = ideal

    def mse(self, training):
        return self._ideal.apply(mean_squared_error,y_pred=training)

    def _max_deviation(self, df_1, df_2):
        diff = (df_1-df_2).abs()

        #with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
        #    print(df_compare)

        return diff.max()*math.sqrt(2)

    def min_value(self, df, diff):
        output = df.apply(min).to_frame("min_value")
        output = output.assign(min_y = df.idxmin())
        output.index.name = "y"
        chosen = self._ideal[output.min_y]
        
        chosen.columns = output.min_y.index
        output["max_deviation"] = self._max_deviation(diff, chosen);
        return output
    
    def match(self, file, lookup, df):
        data_list = []
        lines = FileReader.csv_line_gen(file)
        # Get next() element to skip the header row before iterating
        next(lines)
        for line in lines:
            #print(chosen_func.index)
            match = None
            for index in lookup.index.tolist():
                x = line[0]
                y = float(line[1])
                max_deviation = lookup.max_deviation[index]
                y_caret = df[lookup.min_y[index]][df.index == float(line[0])].values[0]
                y_diff = y - y_caret
                if abs(y_diff) <= max_deviation:
                    match = TestData(x=x, y_test=y, delta_y=y_diff, y_ideal=lookup.min_y[index]) \
                    if (match == None or abs(match.delta_y) > abs(y_diff)) else match
                    #min(matches.items(), key=lambda diff: diff[1]))
            if match:
                # Add match to data_list
                data_list.append(match.to_dict())
        
        return data_list



# SQL

In [89]:
class PersistenceUtil():
    def __init__(self, engine=db.create_engine(f"sqlite:///data/sqlite.db")):
        self._engine = engine

    def df_column_name(self, y_string, replacement):
        return re.sub("y(\d)",replacement,y_string)

    def save_df_to_db(self, df, db, table, regex_column_name):
        df_sql = df.copy()
        df_sql.columns = df_sql.columns.map(lambda c: self._sql_column(c, regex_column_name, "y(\d)"))
        df_sql.index.names = ['X']

        return df_sql.to_sql(table, db, if_exists="replace", index=True)
    
    def create_table(self, table, columns):
        # Get MetaData object
        meta_data = db.MetaData()
        
        # Set test data creation script table
        test_table = db.Table(table, meta_data, \
                              *(db.Column(column, columns[column]) for column in columns))

        #db.Column("X (test func)", db.String(5), autoincrement=False, nullable=False), \
        #                       db.Column("Y (test func)", db.Integer, nullable=False), \
        #                       db.Column("Delta Y (test func)", db.Integer, nullable=False), \
        #                       db.Column("No. of ideal func", db.Integer, nullable=False) \
        #                      )

        meta_data.create_all(self._engine)
        
        return table

    def _sql_column(self, original, replacement, pattern):
        return re.sub(pattern,replacement,original)      
        
    def persist(self, table_name, data):
        # Get MetaData object
        meta_data = db.MetaData()

        table = db.Table(table_name, meta_data, autoload=True, autoload_with=self._engine)
        sql_query = db.insert(table)
        
        connection = self._engine.connect()

        # execute the insert statement
        result = connection.execute(sql_query, data)

        connection.close()
    
    @property
    def engine(self):
        return self._engine

    @engine.setter
    def engine(self, delta_y):
        self._engine = engine

    @engine.deleter
    def engine(self):
        del self._engine



In [69]:
class TestData():
    def __init__(self, x, y_test, delta_y, y_ideal):
        self._x = x
        self._y_test = y_test
        self._delta_y = delta_y
        self._y_ideal = y_ideal

    def __str__(self):
        return str(self.to_dict())

    def to_dict(self):
        return {"X (test func)": self._x, "Y (test func)": self._y_test, \
                "Delta Y (test func)": self._delta_y, "No. of ideal func": self._y_ideal}

    @property
    def delta_y(self):
        return self._delta_y

    @delta_y.setter
    def delta_y(self, delta_y):
        self._delta_y = delta_y

    @delta_y.deleter
    def delta_y(self):
        del self._delta_y

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, x):
        self._x = x

    @x.deleter
    def x(self):
        del self._x


In [93]:
# All data must be visualized logically
# Finally, the training data, the test data, the chosen ideal functions as well as the corresponding /
# assigned datasets are visualized under an appropriately chosen representation of the deviation.

# Where possible, create/compile suitable unit-test

# Its design is sensibly object-oriented
# It includes at least one inheritance
# It includes standard- und user-defined exception handlings
# For logical reasons, it makes use of Pandas’ packages as well as data visualization via Bokeh, sqlalchemy, \
# as well as others
# Write unit-tests for all useful elements
# Your code needs to be documented in its entirety and also include Documentation Strings, known as ”docstrings“

def main():
    '''
    Initialises the logger.
    '''
    logging.basicConfig(filename="app.log", filemode="a", \
                        format="[%(asctime)s] %(name)-6s - %(levelname)-8s - %(message)s", datefmt='%Y-%m-%d %H:%M:%S')

    global logger
    logger = logging.getLogger()
    logger.setLevel(log_level)

    logger.info("App started")

    # Test the logger in DEBUG and NOTSET mode.
    if logger.level < logging.INFO:
        logger.debug("Debugging")
        logger.warning("Test logger")

    df_train = FileReader.read_csv_to_df("data/train.csv")
    df_ideal = FileReader.read_csv_to_df("data/ideal.csv")

    regression = Regression(df_ideal)
    df_train_mse = df_train.apply(regression.mse)

    df_result = regression.min_value(df_train_mse, df_train)

    pu = PersistenceUtil()
    
    pu.save_df_to_db(df_train, pu.engine, "train", r"Y\1 (training func)")
    pu.save_df_to_db(df_ideal, pu.engine, "ideal", r"Y\1 (ideal func)")
    
    
    chosen_ideal = df_ideal[df_result.min_y.tolist()]

    data = regression.match("data/test.csv", df_result, chosen_ideal)


    test_table = pu.create_table("test", {"X (test func)": db.String(5), "Y (test func)": db.Integer, \
                         "Delta Y (test func)": db.Integer, "No. of ideal func": db.Integer})
    
    pu.persist(test_table, data)
    print("Done")

if __name__ == '__main__':
    main()

Done
